# Week 4, Day 3: A neuron as a dot product: receptive fields and cosine tuning

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w004/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gabor(size=21, theta=0.0, wavelength=8.0, sigma=4.0):
    """A Gabor receptive field: stripes at angle theta (radians) under a Gaussian window."""
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1]
    xr = x * np.cos(theta) + y * np.sin(theta)
    return np.exp(-(x ** 2 + y ** 2) / (2 * sigma ** 2)) * np.cos(2 * np.pi * xr / wavelength)

def grating(size=21, theta=0.0, wavelength=8.0):
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1]
    return np.cos(2 * np.pi * (x * np.cos(theta) + y * np.sin(theta)) / wavelength)

rf = gabor(theta=np.radians(30))
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
for ax, (img, t) in zip(axes, [(rf, "receptive field"), (grating(theta=np.radians(30)), "grating 30°"),
                               (grating(theta=np.radians(120)), "grating 120°"), (-grating(theta=np.radians(30)), "grating 30°, inverted")]):
    ax.imshow(img, cmap="RdBu_r", vmin=-1, vmax=1); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

for name, img in [("30°", grating(theta=np.radians(30))), ("120°", grating(theta=np.radians(120))), ("30° inverted", -grating(theta=np.radians(30)))]:
    print(f"{name:>13}: w · x = {rf.ravel() @ img.ravel():8.2f}")

### Exercise: An orientation tuning curve from dot products

For 60 orientations from 0 to 180° (`thetas`, in radians), compute `responses`: the rectified response $\max(0, \mathbf{w} \cdot \mathbf{x})$ of the receptive field `rf` to a grating at each orientation. Plot the tuning curve. Where does it peak, and how wide is it?

In [ ]:
thetas = np.linspace(0, np.pi, 60, endpoint=False)
responses = ...
plt.figure(figsize=(5, 3))
plt.plot(np.degrees(thetas), responses)
plt.xlabel("grating orientation (deg)"); plt.ylabel("response (a.u.)")
plt.show()

In [ ]:
import numpy as np
def test_shape():
    "responses has one value per orientation"
    assert responses.shape == thetas.shape

def test_values():
    "each response is the rectified dot product of the receptive field and the grating"
    for th, r in zip(thetas[::6], responses[::6]):
        expect = max(0.0, rf.ravel() @ grating(theta=th).ravel())
        assert abs(r - expect) < 1e-9

def test_peak():
    "the preferred orientation is 30 degrees"
    assert abs(np.degrees(thetas[np.argmax(responses)]) - 30) < 3

test_shape()
test_values()
test_peak()
print("All checks passed")

### Exercise: Find the preferred direction

Write `preferred_direction(dirs, rates)` that returns the preferred direction in degrees (0 to 360): sum the unit vectors $(\cos\phi, \sin\phi)$ weighted by the rates, and take the angle of the sum with `np.arctan2(y, x)`. Then apply it to the noisy neuron below and store the result in `est_noisy`.

In [ ]:
def preferred_direction(dirs, rates):
    x = ...
    y = ...
    return np.degrees(np.arctan2(y, x)) % 360

dirs = np.radians(np.arange(0, 360, 45))
rng = np.random.default_rng(4)
rates = rng.poisson((20 + 15 * np.cos(dirs - np.radians(135))) * 1.0)    # one second per direction
est_noisy = preferred_direction(dirs, rates)
print("rates:", rates, " estimated preferred direction:", round(est_noisy, 1), "degrees (true 135)")

In [ ]:
import numpy as np
def test_pd():
    "preferred_direction recovers the angle of a cosine-tuned neuron"
    dirs = np.radians(np.arange(0, 360, 45))
    for true in [10, 100, 250]:
        r = 20 + 15 * np.cos(dirs - np.radians(true))
        est = preferred_direction(dirs, r)
        assert abs((est - true + 180) % 360 - 180) < 1e-6, f"true {true}, got {est}"

def test_noisy():
    "est_noisy is within 15 degrees for the noisy neuron"
    assert abs((est_noisy - 135 + 180) % 360 - 180) < 15

test_pd()
test_noisy()
print("All checks passed")